In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    roc_curve,
    average_precision_score
)

import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import shap
import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv('../data/home_vehicle_loans.csv')

# -----------------------------
# Label Encoding
# -----------------------------
le_emp = LabelEncoder()
le_city = LabelEncoder()
le_type = LabelEncoder()

df['emp_enc'] = le_emp.fit_transform(df['employment'])
df['city_enc'] = le_city.fit_transform(df['city_tier'])
df['type_enc'] = le_type.fit_transform(df['loan_type'])

# -----------------------------
# Feature Engineering
# -----------------------------
df['loan_income_ratio'] = (
    df['loan_amt'] / (df['income_mo'] * 12 + 1)
)

df['high_ltv'] = (df['ltv'] > 0.85).astype(int)

df['high_emi_stress'] = (
    df['emi_income'] > 0.40
).astype(int)

df['bad_cibil'] = (
    df['cibil'] < 650
).astype(int)

df['good_cibil'] = (
    df['cibil'] > 750
).astype(int)

df['has_delinquency'] = (
    df['delinq_hist'] > 0
).astype(int)

df['age_income_ratio'] = (
    df['age'] / (df['income_mo'] + 1)
)

# -----------------------------
# Features
# -----------------------------
FEATURES = [
    'age',
    'income_mo',
    'emp_enc',
    'emp_years',
    'cibil',
    'loan_amt',
    'ltv',
    'tenure_mo',
    'interest_rate',
    'emi',
    'emi_income',
    'delinq_hist',
    'num_loans',
    'co_applicant',
    'city_enc',
    'type_enc',
    'loan_income_ratio',
    'high_ltv',
    'high_emi_stress',
    'bad_cibil',
    'good_cibil',
    'has_delinquency',
    'age_income_ratio'
]

TARGET = 'defaulted'

X = df[FEATURES]
y = df[TARGET]

# -----------------------------
# Train Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

# -----------------------------
# LightGBM Model
# -----------------------------
lgb_model = lgb.LGBMClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# -----------------------------
# Train
# -----------------------------
lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(40, verbose=False),
        lgb.log_evaluation(100)
    ]
)

# -----------------------------
# Predictions
# -----------------------------
y_proba = lgb_model.predict_proba(X_test)[:,1]

y_pred = lgb_model.predict(X_test)

# -----------------------------
# Metrics
# -----------------------------
auc = roc_auc_score(y_test, y_proba)

auc_pr = average_precision_score(y_test, y_proba)

print(f"AUC-ROC: {auc:.4f}")

print(f"AUC-PR: {auc_pr:.4f}")

print()

print(
    classification_report(
        y_test,
        y_pred,
        target_names=['No Default','Default']
    )
)

D:\ slice home vehicle loan\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (64000, 23)
Test: (16000, 23)
[100]	valid_0's binary_logloss: 0.582716
[200]	valid_0's binary_logloss: 0.572474
[300]	valid_0's binary_logloss: 0.564773
[400]	valid_0's binary_logloss: 0.558726
AUC-ROC: 0.7289
AUC-PR: 0.3519

              precision    recall  f1-score   support

  No Default       0.89      0.70      0.78     13071
     Default       0.32      0.63      0.42      2929

    accuracy                           0.68     16000
   macro avg       0.60      0.66      0.60     16000
weighted avg       0.79      0.68      0.72     16000



In [2]:
from sklearn.metrics import (
    ConfusionMatrixDisplay
)

# -----------------------------
# Evaluation Plots
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

fig.suptitle(
    'LightGBM Home/Vehicle Loan Default Model',
    fontsize=13,
    fontweight='bold'
)

# -----------------------------
# ROC Curve
# -----------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba)

axes[0].plot(
    fpr,
    tpr,
    color='#1D4ED8',
    lw=2,
    label=f'LightGBM (AUC={auc:.4f})'
)

axes[0].plot(
    [0,1],
    [0,1],
    '--',
    color='gray',
    label='Random'
)

axes[0].set_xlabel('False Positive Rate')

axes[0].set_ylabel('True Positive Rate')

axes[0].set_title(
    'ROC Curve',
    fontweight='bold'
)

axes[0].legend()

# -----------------------------
# Confusion Matrix
# -----------------------------
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['No Default','Default'],
    cmap='Blues',
    ax=axes[1]
)

axes[1].set_title(
    'Confusion Matrix',
    fontweight='bold'
)

# -----------------------------
# Probability Distribution
# -----------------------------
axes[2].hist(
    y_proba[y_test==0],
    bins=50,
    alpha=0.6,
    color='green',
    label='No Default',
    density=True
)

axes[2].hist(
    y_proba[y_test==1],
    bins=50,
    alpha=0.6,
    color='red',
    label='Default',
    density=True
)

axes[2].set_title(
    'Predicted Probability Distribution',
    fontweight='bold'
)

axes[2].set_xlabel('Default Probability')

axes[2].legend()

plt.tight_layout()

plt.savefig(
    '../plots/lgbm_evaluation.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/lgbm_evaluation.png")

# -----------------------------
# SHAP Explainability
# -----------------------------
print("Computing SHAP values...")

explainer = shap.TreeExplainer(lgb_model)

X_sample = X_test.iloc[:500]

shap_values = explainer.shap_values(X_sample)

sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))

shap.summary_plot(
    sv,
    X_sample,
    feature_names=FEATURES,
    plot_type='bar',
    show=False
)

plt.title(
    'Feature Importance — SHAP (LightGBM)',
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    '../plots/shap_lgbm.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/shap_lgbm.png")

# -----------------------------
# Save Models
# -----------------------------
joblib.dump(
    lgb_model,
    '../models/loan_default_model.pkl'
)

joblib.dump(
    le_emp,
    '../models/le_emp.pkl'
)

joblib.dump(
    le_city,
    '../models/le_city.pkl'
)

joblib.dump(
    le_type,
    '../models/le_type.pkl'
)

json.dump(
    {
        'auc_roc': round(auc,4),
        'auc_pr': round(auc_pr,4),
        'default_rate': round(float(y.mean()),4),
        'features': FEATURES
    },
    open('../models/lgbm_metrics.json','w'),
    indent=2
)

print("All LightGBM files saved!")

Saved: plots/lgbm_evaluation.png
Computing SHAP values...
Saved: plots/shap_lgbm.png
All LightGBM files saved!
